In [1]:
import os
from pathlib import Path
from datetime import date

import numpy as np
from scipy.io import loadmat
from scipy.signal import resample
from joblib import Parallel, delayed
import mne
from mne.channels import make_dig_montage
import eelbrain


In [5]:
def compute_envelope_rms(wav_dir, n_songs=10, fs=100, target_rms=0.7):
    """
    Compute RMS energy of the acoustic envelope for each stimulus and
    derive a global scaling factor to bring mean RMS to target_rms.

    Parameters
    ----------
    wav_dir    : Path
    n_songs    : int
    fs         : int    Target sampling rate (default 100 Hz).
    target_rms : float  Desired mean RMS after scaling (default 0.7).

    Returns
    -------
    rms_per_song : dict  {song_id -> rms}
    rms_mean     : float
    rms_std      : float
    scale        : float  Multiply every envelope by this before GFNN.
    """
    rms_per_song = {}

    print(f"Envelope RMS energy per stimulus  (target RMS = {target_rms})")
    print("-" * 52)

    for song_id in range(1, n_songs + 1):
        wav      = eelbrain.load.wav(Path(wav_dir) / f'{song_id}.wav')
        envelope = eelbrain.resample(wav.envelope(), fs)
        x        = np.asarray(envelope)

        rms = np.sqrt(np.mean(x ** 2))
        rms_per_song[song_id] = rms
        print(f"  Song {song_id:2d}:  RMS = {rms:.4f}  "
              f"min = {x.min():.4f}  max = {x.max():.4f}")

    rms_values = np.array(list(rms_per_song.values()))
    rms_mean   = rms_values.mean()
    rms_std    = rms_values.std()

    # Scale so that mean RMS across songs lands at target_rms
    scale = target_rms / rms_mean

    print("-" * 52)
    print(f"  Mean RMS : {rms_mean:.4f}")
    print(f"  Std  RMS : {rms_std:.4f}")
    print(f"  Scale    : {scale:.8f}  (× envelope → mean RMS ≈ {target_rms})")

    # Verify
    scaled_rms = rms_values * scale
    print(f"\n  RMS after scaling:")
    for song_id, srms in zip(rms_per_song.keys(), scaled_rms):
        print(f"    Song {song_id:2d}: {srms:.4f}")
    print(f"  Mean: {scaled_rms.mean():.4f}  Std: {scaled_rms.std():.4f}")

    return rms_per_song, rms_mean, rms_std, scale

In [6]:
BASE_DIR  = Path('NRT').resolve().parent
DATA_ROOT = BASE_DIR / '../TRF/liberi_dataset/doi_10_5061_dryad_g1jwstqmh__v20211008'
WAV_DIR   = DATA_ROOT / 'diliBach_wav_4dryad'

In [8]:
rms_per_song, rms_mean, rms_std, scale = compute_envelope_rms(WAV_DIR)

Envelope RMS energy per stimulus  (target RMS = 0.7)
----------------------------------------------------
  Song  1:  RMS = 2240.2053  min = -231.3876  max = 3542.9392
  Song  2:  RMS = 1847.9188  min = -156.4852  max = 3558.1318
  Song  3:  RMS = 2406.3066  min = -245.7496  max = 3542.3324
  Song  4:  RMS = 2397.5570  min = -230.7142  max = 3552.3803
  Song  5:  RMS = 2433.1965  min = -280.6599  max = 3557.7686
  Song  6:  RMS = 1964.3621  min = -37.9441  max = 3376.1531
  Song  7:  RMS = 2390.5816  min = -264.2867  max = 3558.3814
  Song  8:  RMS = 2041.3427  min = -229.9745  max = 3550.6057
  Song  9:  RMS = 1571.1058  min = -161.0152  max = 2399.5457
  Song 10:  RMS = 2392.9220  min = -270.8410  max = 3543.3437
----------------------------------------------------
  Mean RMS : 2168.5498
  Std  RMS : 283.0908
  Scale    : 0.00032280  (× envelope → mean RMS ≈ 0.7)

  RMS after scaling:
    Song  1: 0.7231
    Song  2: 0.5965
    Song  3: 0.7767
    Song  4: 0.7739
    Song  5: 0.7854
